In [ ]:
# @title <b><font color="orange">WebUI Installer</font></b> {"display-mode":"form"}

Webui = "Forge-Neo" # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Civitai__Key = "" # @param {type:"string", placeholder:"Your Civitai API Key (required)"}
HF_Read_Token = "" # @param {type:"string", placeholder:"Your Hugging Face READ Token (optional)"}
Mount_GDrive = "No" # @param ["No", "Yes"]
Setup_Parallel_Download = True # @param {type:"boolean"}
Setup_Max_Workers = 3 # @param {type:"slider", min:1, max:10, step:1}

from IPython.display import HTML, display

display(HTML('<a href="https://civitai.com/user/account" target="_blank">Get Civitai key</a> &nbsp;|&nbsp; <a href="https://huggingface.co/settings/tokens" target="_blank">Get Hugging Face token</a>'))

setup_parallel_flag = "--setup_parallel_download" if Setup_Parallel_Download else ""

if Mount_GDrive == "Yes":
    from google.colab import drive
    drive.mount("/content/drive")

!curl -sLo /content/setup.py https://github.com/n3iKos/segsmaker-fast/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai__Key" --hf_read_token="$HF_Read_Token" $setup_parallel_flag --setup_max_workers="$Setup_Max_Workers"

if Mount_GDrive == "Yes":
    from pathlib import Path

    drive_root = Path("/content/drive/MyDrive/Segsmaker")
    drive_root.mkdir(parents=True, exist_ok=True)

    for name, path in {"checkpoint": CKPT, "lora": LORA, "vae": VAE, "embeddings": Embeddings}.items():
        target = drive_root / name
        target.mkdir(parents=True, exist_ok=True)
        link = path / f"drive-{name}"
        if not link.exists():
            link.symlink_to(target, target_is_directory=True)

    !rm -rf $WebUI_Output
    output_target = drive_root / {"ComfyUI": "comfyui-output", "SwarmUI": "swarmui-output"}.get(Webui, "output")
    output_target.mkdir(parents=True, exist_ok=True)
    WebUI_Output.symlink_to(output_target, target_is_directory=True)

    if Webui not in {"ComfyUI", "SwarmUI"}:
        webui_cache = WebUI / "cache"
        !rm -rf $webui_cache
        cache_target = drive_root / "cache"
        cache_target.mkdir(parents=True, exist_ok=True)
        webui_cache.symlink_to(cache_target, target_is_directory=True)


In [ ]:
# @title Model Downloader - 5 Checkpoint + 5 LoRA + VAE {"display-mode":"form"}

Checkpoint_1 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Checkpoint_2 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Checkpoint_3 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Checkpoint_4 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Checkpoint_5 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Lora_1 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Lora_2 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Lora_3 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Lora_4 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Lora_5 = "" # @param {type:"string", placeholder:"URL or leave empty"}
VAE_URL = "" # @param {type:"string", placeholder:"URL or leave empty"}
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:10, step:1}

from pathlib import Path
from nenen88 import download_many


def with_target(urls, target):
    target = Path(target)
    return [f"{url.strip()} {target}" for url in urls if url and url.strip()]

checkpoint_urls = [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]
lora_urls = [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]
vae_urls = [VAE_URL]

download_many(with_target(checkpoint_urls, CKPT), parallel=Parallel_Download, max_workers=Max_Workers, label="Checkpoint")
download_many(with_target(lora_urls, LORA), parallel=Parallel_Download, max_workers=Max_Workers, label="LoRA")
download_many(with_target(vae_urls, VAE), parallel=Parallel_Download, max_workers=Max_Workers, label="VAE")


In [ ]:
# @title Extra Assets - Extensions, Embeddings, Upscalers {"display-mode":"form"}

Extension_1 = "" # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_2 = "" # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_3 = "" # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_4 = "" # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_5 = "" # @param {type:"string", placeholder:"git clone URL or leave empty"}
Embedding_1 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_2 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_3 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_1 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_2 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_3 = "" # @param {type:"string", placeholder:"URL or leave empty"}
Assets_Parallel_Download = True # @param {type:"boolean"}
Assets_Max_Workers = 3 # @param {type:"slider", min:1, max:10, step:1}

import os
import shlex
import subprocess
from pathlib import Path
from nenen88 import download_many


def clean_items(values):
    return [value.strip() for value in values if value and value.strip()]


def with_target(urls, target):
    target = Path(target)
    return [f"{url} {target}" for url in clean_items(urls)]

extensions = clean_items([Extension_1, Extension_2, Extension_3, Extension_4, Extension_5])
if extensions:
    Path(Extensions).mkdir(parents=True, exist_ok=True)
    os.chdir(Extensions)
    for repo in extensions:
        subprocess.run(shlex.split(f"git clone {repo}"), check=False)

embedding_urls = [Embedding_1, Embedding_2, Embedding_3]
upscaler_urls = [Upscaler_1, Upscaler_2, Upscaler_3]

download_many(with_target(embedding_urls, Embeddings), parallel=Assets_Parallel_Download, max_workers=Assets_Max_Workers, label="Embeddings")
download_many(with_target(upscaler_urls, Upscalers), parallel=Assets_Parallel_Download, max_workers=Assets_Max_Workers, label="Upscalers")


In [ ]:
# @title FLUX Model Downloader {"display-mode":"form"}

FLUX_Variant = "FLUX.1-schnell (Fast, 4-step)" # @param ["FLUX.1-schnell (Fast, 4-step)", "FLUX.1-dev (Quality, 20-step)"]
FLUX_Unet = "" # @param {type:"string", placeholder:"URL or leave empty"}
FLUX_Clip_L = "" # @param {type:"string", placeholder:"URL or leave empty"}
FLUX_T5XXL = "" # @param {type:"string", placeholder:"URL or leave empty"}
FLUX_VAE = "" # @param {type:"string", placeholder:"URL or leave empty"}
Parallel_FLUX_Download = True # @param {type:"boolean"}
FLUX_Max_Workers = 2 # @param {type:"slider", min:1, max:6, step:1}

from pathlib import Path
from nenen88 import download_many


def add_target(url, target):
    if not url or not url.strip():
        return None
    return f"{url.strip()} {Path(target)}"

text_encoder_dir = globals().get("TE", globals().get("CLIP"))
flux_items = [
    add_target(FLUX_Unet, UNET),
    add_target(FLUX_Clip_L, CLIP),
    add_target(FLUX_T5XXL, text_encoder_dir),
    add_target(FLUX_VAE, VAE),
]

download_many(flux_items, parallel=Parallel_FLUX_Download, max_workers=FLUX_Max_Workers, label="FLUX")


In [ ]:
''' Controlnet '''
%run $Controlnet_Widget


In [ ]:
# @title Launcher WebUI {"display-mode":"form"}

Software = "Forge-Neo" # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Ngrok_Token = "" # @param {type:"string", placeholder:"optional"}
Zrok_Token = "" # @param {type:"string", placeholder:"optional"}
Extra_Args = "" # @param {type:"string", placeholder:"additional launch args"}
Skip_ComfyUI_Check = False # @param {type:"boolean"}
Skip_Widget = False # @param {type:"boolean"}

import shlex

print("Select the same WebUI that you installed in the first cell.")

default_args = {
    "A1111": "--xformers",
    "Forge": "--disable-xformers --opt-sdp-attention --cuda-stream",
    "ReForge": "--xformers --cuda-stream",
    "ReForge-old": "--xformers --cuda-stream",
    "Forge-Classic": "--xformers --cuda-stream --persistent-patches",
    "Forge-Neo": "--xformers --cuda-malloc --cuda-stream",
    "ComfyUI": "--dont-print-server --use-pytorch-cross-attention",
    "SwarmUI": "--launch_mode none",
}

args = []
args.extend(shlex.split(default_args.get(Software, "")))
if Extra_Args.strip():
    args.extend(shlex.split(Extra_Args.strip()))
if Ngrok_Token.strip():
    args.append(f"--N={Ngrok_Token.strip()}")
if Zrok_Token.strip():
    args.append(f"--Z={Zrok_Token.strip()}")
if Skip_ComfyUI_Check:
    args.append("--skip-comfyui-check")
if Skip_Widget:
    args.append("--skip-widget")

launch_args = " ".join(shlex.quote(arg) for arg in args)
%cd -q $WebUI
%run segsmaker.py $launch_args
